# Document Splitting

In [ ]:
# 基础环境准备：加载 API Key 等配置（本 notebook 后面大部分代码其实不需要真实调用 OpenAI，
# 只有涉及 PyPDFLoader/NotionDirectoryLoader 加载课程数据的地方才会用到，本地没有数据文件时可以跳过那几个 cell）
import os
import openai
import sys
sys.path.append('../..')
from dotenv import load_dotenv,find_dotenv
_=load_dotenv(find_dotenv())
openai.api_key=os.environ['OPENAI_API_KEY']

In [ ]:
# 文本切分器（Text Splitter）：RAG 流程第二步——文档切分（Document Splitting）
# 把加载进来的长文档切成一个个小块（chunk），因为：
#  1）Embedding 模型和 LLM 的输入长度有限；
#  2）检索时希望返回和问题最相关的“小段”而不是整篇长文档，减少无关噪音。
#
# 注意：这里是新版路径 langchain_text_splitters（独立子包）。
# 旧版课程写法是 `from langchain.text_splitter import ...`，
# 新版 langchain 1.4.0 下文本切分器已经拆分到独立的 langchain_text_splitters 包，旧路径会 ModuleNotFoundError。
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

In [ ]:
# chunk_size：每个切分块的最大长度（这里的度量单位是字符数）
# chunk_overlap：相邻两个块之间重叠的长度，用来避免关键信息正好被切在块的边界上导致语义丢失
chunk_size=26
chunk_overlap=4

In [ ]:
# RecursiveCharacterTextSplitter：推荐的默认切分器。它会按一组分隔符（默认 ["\n\n", "\n", " ", ""]）
# 从大到小依次尝试切分，尽量保持段落/句子的完整性，是最常用的通用文本切分策略。
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)
# CharacterTextSplitter：更简单的切分器，只按照一个固定的分隔符（默认是 "\n\n"）切分，
# 如果文本里根本没有出现这个分隔符，它就不会切分，即使超过 chunk_size 也会保留成一个大块。
c_splitter = CharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

In [ ]:
# 26 个字母，长度正好等于 chunk_size=26，所以理论上不会被切分
text1 = 'abcdefghijklmnopqrstuvwxyz'

In [ ]:
# 实测结果：['abcdefghijklmnopqrstuvwxyz']，正好一整块，没有被切分
r_splitter.split_text(text1)

In [ ]:
# 34 个字符，超过了 chunk_size=26，会被切成两块，且两块之间有 chunk_overlap=4 个字符重叠
text2 = 'abcdefghijklmnopqrstuvwxyzabcdefg'

In [ ]:
# 实测结果：['abcdefghijklmnopqrstuvwxyz', 'wxyzabcdefg']
# 第二块开头的 "wxyz" 就是和第一块末尾重叠的 4 个字符（chunk_overlap=4）
r_splitter.split_text(text2)

In [ ]:
# 用空格分隔的字母，长度是 53，用来对比 RecursiveCharacterTextSplitter 和 CharacterTextSplitter 的行为差异
text3 = "a b c d e f g h i j k l m n o p q r s t u v w x y z"

In [ ]:
# RecursiveCharacterTextSplitter 默认分隔符列表里包含 " "（空格），所以能正确按空格切成多块
# 实测结果：['a b c d e f g h i j k l m', 'l m n o p q r s t u v w x', 'w x y z']
r_splitter.split_text(text3)

In [ ]:
# CharacterTextSplitter 默认分隔符是 "\n\n"，text3 里根本没有换行符，所以切分失败，
# 实测结果：整段文字被当成一整块返回（会打印一条 "created a chunk of size 53, which is longer than the specified 26" 的警告，
# 这正是这个例子想说明的问题：CharacterTextSplitter 找不到分隔符时不会强制切分）
c_splitter.split_text(text3)

In [ ]:
# 显式把分隔符指定为空格 " "，这样 CharacterTextSplitter 就能正确切分了
# 实测结果和 r_splitter 一致：['a b c d e f g h i j k l m', 'l m n o p q r s t u v w x', 'w x y z']
c_splitter = CharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separator = ' '
)
c_splitter.split_text(text3)

## Recursive splitting details

In [ ]:
# 一段更接近真实场景的文本：包含段落（用两个换行符分隔）、句子（用句号+空格分隔）、单词（用空格分隔）
# 用来演示"层级式"的递归切分策略：先尝试按段落切，段落太长再按句子切，句子太长再按单词切
some_text = """When writing documents, writers will use document structure to group content. \
This can convey to the reader, which idea's are related. For example, closely related ideas \
are in sentances. Similar ideas are in paragraphs. Paragraphs form a document. \n\n  \
Paragraphs are often delimited with a carriage return or two carriage returns. \
Carriage returns are the "backslash n" you see embedded in this string. \
Sentences have a period at the end, but also, have a space.\
and words are separated by space."""

In [ ]:
# 看一下这段文字总共多少字符，方便和后面 chunk_size=450 对比
len(some_text)

In [ ]:
# chunk_overlap=0：这里先不考虑重叠，专注对比两种切分器对同一段文字的处理差异
c_splitter = CharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=0,
    separator = ' '
)
# 显式传入 separators 列表，优先级从高到低：先按段落（\n\n）切，再按句子内换行（\n）切，
# 再按空格切，最后按字符切（""）——这就是"递归"的含义
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=0,
    separators=["\n\n", "\n", " ", ""]
)

In [ ]:
# CharacterTextSplitter 只会按单一分隔符（空格）切，不管段落结构，所以切出来的块可能会把一个段落硬生生切断
c_splitter.split_text(some_text)

In [ ]:
# RecursiveCharacterTextSplitter 优先按 "\n\n" 切，所以能把两个自然段整齐地分成两块
r_splitter.split_text(some_text)

In [ ]:
# 把 chunk_size 缩小到 150，并尝试用 "\. "（句号+空格）作为句子级分隔符
#
# 【版本差异+真实验证】新版 langchain_text_splitters 里 RecursiveCharacterTextSplitter 的
# is_separator_regex 参数默认值是 False（旧版课程年代实际相当于 True，separators 直接当正则用）。
# 如果不显式传 is_separator_regex=True，"\. " 会被当成字面字符串（还会被 re.escape 转义），
# 根本匹配不到文本里真正的 "句号+空格"，导致这个"反面教材"在新版本下悄悄失效、看不出问题
# （我实际跑过：不加这个参数时，输出和下一格用 lookbehind 修复后的结果一模一样，根本演示不出 bug）。
# 所以这里显式加上 is_separator_regex=True，还原课程原本想展示的效果：
# 普通字符串分隔符匹配后会把分隔符本身也吃掉，导致句子丢失末尾的句号，句号跑到了下一块的开头。
r_splitter=RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=0,
    separators=["\n\n", "\n", "\. ", " ", ""],
    is_separator_regex=True,
)
r_splitter.split_text(some_text)

In [ ]:
# 用正则的"零宽度断言" (?<=\. ) 代替普通的 "\. "：
# 它表示"在句号+空格之后切开，但不消耗这几个字符"，这样句号就会保留在前一句的末尾，不会丢失。
# 同样必须显式传 is_separator_regex=True，否则这个分隔符也会被当成字面字符串处理，不会按正则的
# lookbehind 语义生效（原因同上一格：新版默认 is_separator_regex=False）。
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=0,
    separators=["\n\n", "\n", "(?<=\. )", " ", ""],
    is_separator_regex=True,
)
r_splitter.split_text(some_text)

In [ ]:
# 加载真实 PDF 文档，为下面的“对文档切分”做准备（新版路径 langchain_community.document_loaders，见 01 节说明）
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("docs/cs229_lectures/MachineLearning-Lecture01.pdf")
pages = loader.load()

In [ ]:
# 对真实 PDF 文档使用 CharacterTextSplitter：按换行符 "\n" 切分，chunk_size=1000（字符数）
# length_function=len 指定用 Python 内置的 len() 计算长度（默认就是它，这里显式写出来只是为了说明这个参数的作用：
# 也可以换成基于 token 数的计数函数，让 chunk_size 按 token 而不是字符数来衡量）
from langchain_text_splitters import CharacterTextSplitter
text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len
)

In [ ]:
# split_documents 会保留每个 Document 原来的 metadata（比如 source、page），只是把 page_content 切成更小的块，
# 每个切出来的小块仍然是一个独立的 Document 对象
docs=text_splitter.split_documents(pages)

In [ ]:
# 切分后的块数，通常会比原始页数多（因为长页面会被拆成多块）
len(docs)

In [ ]:
# 原始 PDF 页数，和上面 len(docs) 对比可以看出切分后块数变多了
len(pages)

In [ ]:
# 再用 Notion 笔记数据做一次同样的切分演示
from langchain_community.document_loaders import NotionDirectoryLoader
loader = NotionDirectoryLoader("docs/Notion_DB")
notion_db = loader.load()

In [ ]:
docs=text_splitter.split_documents(notion_db)

In [ ]:
len(notion_db)

In [ ]:
len(docs)

## Token splitting
按 Token（大模型实际计费/计数的单位，一般不等于字符数或单词数）而不是字符数来切分，
这样切出来的块更贴近 LLM 真实看到的输入长度限制，避免因为多语言/特殊符号导致字符数和 token 数偏差太大。

In [ ]:
# TokenTextSplitter 底层用 tiktoken（OpenAI 的分词库）把文本编码成 token，再按 token 数切分
from langchain_text_splitters import TokenTextSplitter

In [ ]:
# chunk_size=1 表示每块只放 1 个 token，用极端小的例子来直观展示"token 切分"和"字符切分"的区别
text_splitter = TokenTextSplitter(chunk_size=1, chunk_overlap=0)

In [ ]:
text1 = "foo bar bazzyfoo"

In [ ]:
# 实测结果：['foo', ' bar', ' b', 'az', 'zy', 'foo']
# 可以看到 token 切分不是按空格或字符切的，"bazzyfoo" 这个生造词被 tiktoken 拆成了 'b'+'az'+'zy'+'foo' 好几个 token，
# 说明 token 粒度和人类直觉的"单词"是不一致的
text_splitter.split_text(text1)

In [ ]:
# 换成更实用的 chunk_size=10（每块 10 个 token），准备对真实 PDF 文档做 token 切分
text_splitter = TokenTextSplitter(chunk_size=10, chunk_overlap=0)

In [ ]:
docs = text_splitter.split_documents(pages)

In [ ]:
docs[0]

In [ ]:
# split_documents 会把原始 Document 的 metadata 原样复制到每个切出来的小块上
pages[0].metadata

## Context aware splitting
前面的切分方式都只关心长度，不关心文档结构。对于 Markdown 这类有明确层级结构（# 标题）的文档，
更好的做法是先按标题层级切分，并把标题信息保留到每个块的 metadata 里，这样后续检索时能知道某段内容来自哪个章节。

In [ ]:
from langchain_community.document_loaders import NotionDirectoryLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter

In [ ]:
# 构造一个带多级标题（#、##、###）的示例 Markdown 文档
markdown_document = """# Title\n\n \
## Chapter 1\n\n \
Hi this is Jim\n\n Hi this is Joe\n\n \
### Section \n\n \
Hi this is Lance \n\n
## Chapter 2\n\n \
Hi this is Molly"""

In [ ]:
# 告诉 splitter：遇到 "#" 就记录为 Header 1，遇到 "##" 记录为 Header 2，遇到 "###" 记录为 Header 3
# 这些标题文字最终会被写进每个切分块的 metadata
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

In [ ]:
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)
md_header_splits = markdown_splitter.split_text(markdown_document)

In [ ]:
# 实测：Document(metadata={'Header 1': 'Title', 'Header 2': 'Chapter 1'}, page_content='Hi this is Jim  \nHi this is Joe')
# 可以看到这个块的 metadata 里带上了它所属的 Header 1 / Header 2 标题路径
md_header_splits[0]

In [ ]:
# 第二个块进入了 "### Section" 这一级，所以 metadata 里多了 Header 3
md_header_splits[1]

In [ ]:
# 把 Notion 笔记（真实数据）的所有文档内容拼接成一整段文本，再用 Markdown 标题切分
loader = NotionDirectoryLoader("docs/Notion_DB")
docs = loader.load()
txt = ' '.join([d.page_content for d in docs])

In [ ]:
# 这次只关心两级标题
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

In [ ]:
md_header_splits = markdown_splitter.split_text(txt)

In [ ]:
md_header_splits[0]